# 🐍 Python Sets: The Definitive Interview Guide & Reference

This notebook provides an exhaustive, production-grade reference and interview guide for Python sets (`set` and `frozenset`). It is structured systematically to cover fundamental characteristics, under-the-hood CPython implementations, time and space complexities, all built-in methods, set operators, custom hashable elements, edge cases, and high-frequency interview patterns.

---

## 🎯 1. Definition and Fundamental Characteristics

A Python set is an **unordered**, **mutable**, **dynamic** collection of **unique**, **hashable** elements.

### 🔑 Core Properties for Interviews
1. **Uniqueness:** Duplicates are disallowed. Adding an existing item results in a no-op.
2. **Unordered:** Sets do not maintain any insertion order or indexed positions. Operations like index lookups (`s[0]`) or slicing (`s[1:3]`) raise a `TypeError`.
3. **Mutable Container:** Elements can be added or removed in-place. Its immutable counterpart is the `frozenset`.
4. **Element Constraints (Hashability):** All elements in a set **must be hashable** (must implement stable `__hash__()` and `__eq__()` methods). Thus, mutable objects like `list`, `set`, or `dict` cannot be elements of a set.
5. **Associative Hash Map Under-the-Hood:** Sets are built using a hash table where elements act as the table keys (without storing associated values).

### 🛠️ Initializing Sets
```python
# 1. Direct Literal Initialization (using curly braces)
s1 = {1, 2, 3}

# 2. set() Constructor (necessary for empty sets!)
# Warning: s = {} creates an empty dictionary, not an empty set.
empty_set = set()
s2 = set([1, 2, 3, 2]) # Result: {1, 2, 3} (duplicates removed)

# 3. Set Comprehension
s3 = {x for x in range(10) if x % 2 == 0}
```

In [ ]:
# Demonstrating set creation and duplicate behavior
s_literal = {1, 2, 3, 3, 2, 1}
s_empty = set()
s_empty_dict_mistake = {} # This is a dict!
s_comp = {x**2 for x in [-2, -1, 0, 1, 2]}

print("Literal unique values:", s_literal)
print("Type of s_empty (set()):", type(s_empty))
print("Type of s_empty_dict_mistake ({}):", type(s_empty_dict_mistake))
print("Set Comprehension output (notice squares filter duplicates):", s_comp)

## 🧠 2. Under the Hood: CPython Internals & Hashing Mechanics

Understanding the internal architecture of sets is a key differentiator in advanced interviews.

### 🔍 A. Sets vs. Dictionaries: The Storage Layout
In CPython, a set is represented by the `PySetObject` structure. 
- Unlike dictionaries (which transitioned to a compact dual-array layout in PEP 468), **sets are still implemented as traditional sparse hash tables**.
- The entries in a set hash table are defined by a simple C struct:
  ```c
  typedef struct {
      PyObject *key;   // Pointer to element
      Py_ssize_t hash; // Cached hash code of key
  } setentry;
  ```
- **Why no compact layout for sets?** 
  In a dictionary, the compact layout saves memory because empty slots in a sparse array are small integers (1 byte) instead of full 24-byte entry structs (which store hash, key pointer, and value pointer).
  For sets, each entry struct is only 16 bytes (since there is no value pointer). The CPython developers determined that the overhead of maintaining dual arrays (indices and entries) and the extra redirection costs did not offer enough memory benefits for sets to offset the degradation in performance of key set operations (like union and intersection).
  - **Direct Consequence:** Since sets are built on a traditional sparse array that does not track insertion indices, **sets are fundamentally unordered**.

---

### 🔑 B. Lookup & Probing Algorithm
Set lookup utilizes the same **Open Addressing** with **Pseudo-random Probing** algorithm as dictionaries:
1. Calculates the element hash code.
2. Computes initial bucket: `index = hash_code & (table_size - 1)`.
3. If a collision occurs (different keys mapping to the same bucket index), CPython probes using the recurrence formula:
   `next_index = (5 * current_index + 1 + perturb) % table_size`
   Where `perturb` starts as the hash code and shifts right by 5 bits on each step.

---

### 🗑️ C. Deletion Tombstones
To prevent breaking existing collision probing chains when an item is deleted, CPython does not clear the slot to `NULL`. Instead, it marks the slot with a tombstone sentinel pointer (indicating a dummy/deleted element). Subsequent lookups skip tombstones and continue probing, while subsequent insertions can reclaim and overwrite them.

---

### 📈 D. Resizing and Load Factor
- **Load Factor:** CPython sets resize when the table reaches a load factor threshold of **$3/5$** ($60\%$).
- **Sizing Policy:** The set doubles/quadruples in size (constrained to a power of 2) when resizing is triggered, and all active elements are copied and rehashed, clearing out all dummy tombstones.

In [ ]:
import sys

# Analyzing set memory growth patterns
s = set()
print(f"Empty set memory size: {sys.getsizeof(s)} bytes\n")

prev_size = sys.getsizeof(s)
for i in range(100):
    s.add(i)
    curr_size = sys.getsizeof(s)
    if curr_size != prev_size:
        print(f"Length: {len(s):2d} | Set Memory Size: {curr_size} bytes (Table resized & rehashed!)")
        prev_size = curr_size

## ⏳ 3. Time and Space Complexities

Set operations are mathematically powerful. Interviewers frequently test your knowledge of both single-element and mathematical multi-set time complexities.

### Single-Element Operations

| Operation | Average Case | Worst Case | Mathematical Reason / Explanation |
| :--- | :--- | :--- | :--- |
| **Add (`s.add(x)`)** | $O(1)$ | $O(N)$ | Amortized $O(1)$. Worst case is when the table triggers a resize and copies/rehashes all elements. |
| **Delete (`s.remove(x)`)** | $O(1)$ | $O(N)$ | Finds elements by hash lookup and flags slot with dummy placeholder. |
| **Contains (`x in s`)** | $O(1)$ | $O(N)$ | Performs direct hash table lookup. Highly efficient compared to lists. |

### Multi-Set Mathematical Operations

| Operation | Operator | Time Complexity | Notes / Explanation |
| :--- | :--- | :--- | :--- |
| **Union** | `s1 | s2` | $O(N_1 + N_2)$ | Iterates and copies elements of both sets into a new set. |
| **Intersection** | `s1 & s2` | $O(\min(N_1, N_2))$ | CPython optimization: Iterates over the smaller set and checks membership in the larger set. |
| **Difference** | `s1 - s2` | $O(N_1)$ | Iterates over `s1` and checks if element is not in `s2`. |
| **Symmetric Diff.** | `s1 ^ s2` | $O(N_1 + N_2)$ | Computes elements in either `s1` or `s2`, but not both. |
| **Is Subset** | `s1 <= s2` | $O(N_1)$ | Checks if all elements of `s1` exist in `s2`. |
| **Is Disjoint** | `s1.isdisjoint(s2)`| $O(\min(N_1, N_2))$ | Iterates through the smaller set; stops early at the first common element. |

## 🛠️ 4. The Complete Set API (All 17 Built-In Methods)

Python sets contain exactly **17 built-in methods**. Here is a systematic breakdown of each method with dedicated code examples.

---

### Category A: Single-Element Mutators

#### 1. `s.add(elem)`
- **Behavior:** Adds `elem` to the set.
- **Key Interview Point:** If the element is already present, the set remains unchanged (no-op).

In [ ]:
s = {1, 2}

# 1. Add new element
s.add(3)
print("After adding 3:", s)

# 2. Add duplicate element (no-op)
s.add(2)
print("After adding 2 again (no change):", s)

#### 2. `s.remove(elem)`
- **Behavior:** Removes `elem` from the set.
- **Key Interview Point:** **Raises `KeyError`** if the element is not found in the set.

In [ ]:
s = {1, 2, 3}

# 1. Remove existing element
s.remove(2)
print("After removing 2:", s)

# 2. Remove missing element raises KeyError
try:
    s.remove(99)
except KeyError as e:
    print("remove(99) raised KeyError:", e)

#### 3. `s.discard(elem)`
- **Behavior:** Removes `elem` from the set.
- **Key Interview Point:** **Does NOT raise an error** if the element is not found (safe delete method).

In [ ]:
s = {1, 2, 3}

# 1. Discard existing element
s.discard(2)
print("After discarding 2:", s)

# 2. Discard missing element (no error raised)
s.discard(99)
print("After discarding missing 99 (no change):", s)

#### 4. `s.pop()`
- **Behavior:** Removes and returns an arbitrary element from the set.
- **Key Interview Point:** **Raises `KeyError`** if the set is empty.

In [ ]:
s = {"apple", "banana", "cherry"}

# Pops arbitrary item
val = s.pop()
print(f"Popped item: {val} | Remaining Set: {s}")

# Pop from empty set raises KeyError
empty_s = set()
try:
    empty_s.pop()
except KeyError as e:
    print("Popping from empty set raised KeyError:", e)

#### 5. `s.clear()`
- **Behavior:** Removes all elements from the set in-place.
- **Key Interview Point:** The set container object remains in memory with the same ID, just empty.

In [ ]:
s = {1, 2, 3}
print("Original ID:", id(s))

s.clear()
print("After clear():", s)
print("Same object ID maintained?", id(s))

---

### Category B: Multi-Set Operations (New Set Output)

#### 6. `s.union(*others)` (Operator: `|`)
- **Behavior:** Returns a new set containing all unique elements from `s` and all `others`.
- **Key Interview Point:** The method `.union()` accepts any iterable (list, tuple, etc.). The operator `|` requires all operands to be actual sets.

In [ ]:
s1 = {1, 2}

# 1. Union method with list iterable
u1 = s1.union([3, 4, 3])
print("Union method with list:", u1)

# 2. Union operator with set
u2 = s1 | {3, 4}
print("Union operator with set:", u2)

#### 7. `s.intersection(*others)` (Operator: `&`)
- **Behavior:** Returns a new set containing elements common to `s` and all `others`.
- **Key Interview Point:** The method `.intersection()` accepts any iterable, whereas `&` requires actual sets.

In [ ]:
s1 = {1, 2, 3}

# 1. Intersection method with tuple iterable
i1 = s1.intersection((2, 3, 4))
print("Intersection method with tuple:", i1)

# 2. Intersection operator with set
i2 = s1 & {2, 3, 4}
print("Intersection operator with set:", i2)

#### 8. `s.difference(*others)` (Operator: `-`)
- **Behavior:** Returns a new set with elements in `s` that are not in any of the `others`.
- **Key Interview Point:** The method `.difference()` accepts any iterable, whereas `-` requires actual sets.

In [ ]:
s1 = {1, 2, 3, 4}

# 1. Difference method with list iterable
d1 = s1.difference([2, 3])
print("Difference method with list:", d1)

# 2. Difference operator with set
d2 = s1 - {2, 3}
print("Difference operator with set:", d2)

#### 9. `s.symmetric_difference(other)` (Operator: `^`)
- **Behavior:** Returns a new set with elements in either `s` or `other` but not both.
- **Key Interview Point:** The method accepts any iterable, whereas `^` requires actual sets.

In [ ]:
s1 = {1, 2, 3}

# 1. Symmetric difference method with list
sd1 = s1.symmetric_difference([3, 4, 5])
print("Symmetric difference method with list:", sd1)

# 2. Symmetric difference operator with set
sd2 = s1 ^ {3, 4, 5}
print("Symmetric difference operator with set:", sd2)

---

### Category C: Multi-Set Mutators (In-Place Modifications)

#### 10. `s.update(*others)` (Operator: `|=`)
- **Behavior:** Updates `s` in-place, adding all elements from `others`.
- **Key Interview Point:** Mutates the set container and returns `None`.

In [ ]:
s = {1, 2}

# 1. Update method with list
s.update([3, 4])
print("After update() method:", s)

# 2. Update operator with set
s |= {5, 6}
print("After |= update operator:", s)

#### 11. `s.intersection_update(*others)` (Operator: `&=`)
- **Behavior:** Updates `s` in-place, keeping only elements found in both `s` and `others`.
- **Key Interview Point:** Mutates the set container and returns `None`.

In [ ]:
s = {1, 2, 3}

# 1. Intersection update method with list
s.intersection_update([2, 3, 4])
print("After intersection_update() method:", s)

# 2. Intersection update operator with set
s &= {3, 4, 5}
print("After &= operator:", s)

#### 12. `s.difference_update(*others)` (Operator: `-=`)
- **Behavior:** Updates `s` in-place, removing all elements found in `others`.
- **Key Interview Point:** Mutates the set container and returns `None`.

In [ ]:
s = {1, 2, 3, 4}

# 1. Difference update method with list
s.difference_update([2, 3])
print("After difference_update() method:", s)

# 2. Difference update operator with set
s -= {4}
print("After -= operator:", s)

#### 13. `s.symmetric_difference_update(other)` (Operator: `^=`)
- **Behavior:** Updates `s` in-place, keeping only elements in either `s` or `other` but not both.
- **Key Interview Point:** Mutates the set container and returns `None`.

In [ ]:
s = {1, 2, 3}

# 1. Symmetric difference update method with list
s.symmetric_difference_update([3, 4, 5])
print("After symmetric_difference_update() method:", s)

# 2. Symmetric difference update operator with set
s ^= {1, 9}
print("After ^= operator:", s)

---

### Category D: Set Comparisons & Relations

#### 14. `s.isdisjoint(other)`
- **Behavior:** Returns `True` if `s` has no elements in common with `other`.
- **Key Interview Point:** Stops iterating as soon as the first common element is found ($O(\min(N_1, N_2))$ average).

In [ ]:
s = {1, 2, 3}

# 1. Disjoint checks
print("Disjoint with {4, 5}:", s.isdisjoint({4, 5}))
print("Disjoint with {3, 4}:", s.isdisjoint({3, 4}))

#### 15. `s.issubset(other)` (Operator: `<=`)
- **Behavior:** Returns `True` if all elements of `s` are in `other`.
- **Key Interview Point:** Operator `<=` checks standard subset. Strict subset operator `<` requires `s` to be subset but not equal to `other`.

In [ ]:
s = {1, 2}

# 1. Subset checks with methods
print("Is subset of [1, 2, 3]:", s.issubset([1, 2, 3]))

# 2. Subset checks with operators
print("Subset check (<=):", s <= {1, 2})
print("Strict subset check (<):", s < {1, 2})

#### 16. `s.issuperset(other)` (Operator: `>=`)
- **Behavior:** Returns `True` if `s` contains all elements of `other`.

In [ ]:
s = {1, 2, 3}

# Superset checks
print("Is superset of [1, 2]:", s.issuperset([1, 2]))
print("Superset check (>=):", s >= {1, 2, 3, 4})

---

### Category E: Copying

#### 17. `s.copy()`
- **Behavior:** Returns a **shallow copy** of the set.
- **Key Interview Point:** The elements themselves are not deep-copied; only their references are copied to a new set container.

In [ ]:
s = {1, 2, 3}
s_copy = s.copy()

print("Copy container matches original?", s_copy is s)
print("Copy contents equal?", s_copy == s)

## 🔑 5. Custom Hashable Elements

Just like dictionary keys, any custom object stored in a set must implement:
1. `__hash__()` returning a stable integer.
2. `__eq__()` checking logical equivalence.
3. The invariant: If `a == b`, then `hash(a) == hash(b)`.

If a class defines `__eq__()` but not `__hash__()`, Python sets `__hash__ = None` and raises a `TypeError` when you try to insert it into a set.

In [ ]:
class CustomNode:
    def __init__(self, node_id: int, label: str):
        self.node_id = node_id
        self.label = label

    def __eq__(self, other):
        if not isinstance(other, CustomNode):
            return NotImplemented
        return self.node_id == other.node_id

    def __hash__(self):
        return hash(self.node_id)

    def __repr__(self):
        return f"Node({self.node_id})"

# Testing custom node deduplication in a set
node_set = {CustomNode(1, "A"), CustomNode(1, "B"), CustomNode(2, "C")}
print("Deduplicated nodes (based on node_id):", node_set)

## ⚠️ 6. Common Pitfalls, Gotchas, and Edge Cases

### Gotcha A: The Empty Set Curly Brace Trap
- `{}` initializes a dictionary, not a set. An empty set must be instantiated using `set()`.

### Gotcha B: Modifying a Set During Iteration
- Adding or removing elements while iterating over a set changes the underlying sparse hash table structure, triggering a `RuntimeError: Set size changed during iteration`.
- **Solution:** Loop over `list(s)` or create a new set.

### Gotcha C: Storing Unhashable Mutable Elements
- Attempting to add a list, set, or dictionary directly to a set raises `TypeError: unhashable type`.
- Tuples are only hashable if all of their nested elements are hashable. A tuple containing a list `(1, 2, [3])` is unhashable.

### Gotcha D: Float vs Integer Equality Collisions
- Since `1 == 1.0` is `True` and `hash(1) == hash(1.0)` is equal, a set can contain only one of them. The first one inserted takes precedence.

In [ ]:
# Float and Int Key Collision in Set
collision_set = {1, 1.0}
print("Collision set contents:", collision_set) # Only holds 1!

# Unhashable tuple containing a list
try:
    bad_tup = (1, 2, [3, 4])
    s = {bad_tup}
except TypeError as e:
    print("Caught expected TypeError for nested mutable tuple:", e)

# Modifying set during iteration trap
s = {1, 2, 3}
try:
    for item in s:
        if item % 2 == 0:
            s.remove(item)
except RuntimeError as e:
    print("Modifying set during iteration raised:", e)

## ❄️ 7. Frozen Sets (`frozenset`)

A `frozenset` is an **immutable** and **hashable** version of a set.

### 🔑 Why is `frozenset` needed?
1. **Hashable Sets:** Since a standard `set` is mutable, it is unhashable. If you need to create a **set of sets** or use a **set as a dictionary key**, you must wrap the inner set inside a `frozenset`.
2. **Read-Only API:** It provides only non-mutating mathematical operations (`union()`, `intersection()`, etc.), protecting the container from side-effects.

```python
# Initializing frozensets
fs = frozenset([1, 2, 3])

# Nested sets (Invalid)
nested = { {1, 2}, {3, 4} } # Raises TypeError!

# Nested sets (Valid using frozenset)
nested_ok = { frozenset({1, 2}), frozenset({3, 4}) } # Works!
```

In [ ]:
# Demonstrating frozenset behavior
fs = frozenset([1, 2, 3])
print("Frozenset:", fs)

# Immutable check
try:
    fs.add(4)
except AttributeError as e:
    print("Expected AttributeError when calling .add on frozenset:", e)

# Using frozenset as a key in a dictionary
d = {fs: "Immutable Key Match!"}
print("Dictionary lookup with frozenset key:", d[frozenset([1, 2, 3])])

## 🏆 8. Top Coding Interview Patterns & Practical Use Cases

Sets are essential for optimizing algorithmic searches.

---

### Pattern A: $O(1)$ Deduplication & Seen Check
- **Concept:** Store items in a set to quickly check if a value has been visited or processed in $O(1)$ lookup time.
- **Example:** Visited path tracking in DFS/BFS graph traversals.

---

### Pattern B: Visited Coordinates in Grid Traversals
- **Concept:** BFS/DFS grid algorithms track visited coordinates. Lists are mutable and unhashable, so you store coordinates as tuples `(row, col)` in a `set`.

---

### Pattern C: Two-Pointer Sliding Window Unique Tracker (Longest Substring Without Repeating Characters)
- **Concept:** Maintain a dynamic set representing the current sliding window. As the right pointer moves, elements are checked against the set for duplicates, and the left pointer shrinks the window in $O(1)$ operations.

In [ ]:
# Pattern B: DFS Visited Coordinates Grid simulation
def is_path_valid(grid, start, end):
    rows, cols = len(grid), len(grid[0])
    visited = set()
    
    def dfs(r, c):
        if (r, c) == end:
            return True
        visited.add((r, c))
        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1 and (nr, nc) not in visited:
                if dfs(nr, nc):
                    return True
        return False
        
    return dfs(start[0], start[1])

grid = [
    [1, 1, 0],
    [0, 1, 1],
    [0, 0, 1]
]
print("Is path valid (Grid coordinate tracking):", is_path_valid(grid, (0, 0), (2, 2)))

# Pattern C: Longest Substring Without Repeating Characters
def length_of_longest_substring(s: str) -> int:
    char_set = set()
    left = 0
    max_len = 0
    for right in range(len(s)):
        while s[right] in char_set:
            char_set.remove(s[left])
            left += 1
        char_set.add(s[right])
        max_len = max(max_len, right - left + 1)
    return max_len

print("Longest Substring Without Repeating Characters ('abcabcbb'):", length_of_longest_substring("abcabcbb"))

## 🏁 Summary & Interview Checklist

Before your Python interview, ensure you are solid on these set checkpoints:

- [ ] **Sparse Hash Table Layout:** Understand that sets do not use the compact layout (they are traditional flat sparse tables). This makes sets fundamentally **unordered**.
- [ ] **Collision Probing:** Explain open addressing with pseudo-random probing for collision resolution.
- [ ] **Method vs. Operator Syntax:** Remember that methods (`union()`) accept any iterable, whereas operators (`|`) strictly require sets as both operands.
- [ ] **Uniqueness Invariance:** Understand that numeric equality collides (`1` and `1.0` occupy the same bucket).
- [ ] **Deletion Tombstones:** Explain that CPython uses placeholder tombstone markings to maintain probing chain continuity on deletion.
- [ ] **Frozenset Use Case:** Explain that a `frozenset` is the immutable, hashable variant of a set, allowing for nesting sets (set of sets) or dictionary keys.